# 11 — Function Calling Training

Train the model to correctly invoke `aro ask` tools (function calling).
ARO Coder has 18 tools available via `aro ask`. This notebook generates
training pairs that teach the model:

1. **When** to call each tool (trigger recognition)
2. **How** to format the call (correct JSON arguments)
3. **What** to do with the result (interpret + next step)
4. **Fix patterns** — `aro_check` error → `edit_file` fix → verify

**Input:** Tool definitions from `../../Sources/AROAsk/Tools/`
**Output:** `../data/02_knowledge/knowledge_pairs.jsonl` (appended)

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')

import json, re, random
from pathlib import Path
from collections import Counter

DATA_OUT = DATA_ROOT / '11_function_calling'
DATA_OUT.mkdir(parents=True, exist_ok=True)
print(f'Output: {DATA_OUT}')

## Tool definitions and example calls

In [ ]:
TOOLS = [
    {'name': 'read_file', 'params': ['path', 'offset?', 'limit?'], 'examples': [
        {'user': 'Read the main.aro file', 'args': {'path': 'main.aro'}},
        {'user': 'Show me lines 10-20 of users.aro', 'args': {'path': 'users.aro', 'offset': 10, 'limit': 10}},
    ]},
    {'name': 'write_file', 'params': ['path', 'content'], 'examples': [
        {'user': 'Create a hello world ARO app', 'args': {'path': 'main.aro', 'content': '(Application-Start: Hello) {\n    Log "Hello!" to the <console>.\n    Return an <OK: status> for the <greeting>.\n}'}},
    ]},
    {'name': 'edit_file', 'params': ['path', 'old_string', 'new_string'], 'examples': [
        {'user': 'Fix the string concat in main.aro', 'args': {'path': 'main.aro', 'old_string': '<first> + " " + <last>', 'new_string': '<first> ++ " " ++ <last>'}},
    ]},
    {'name': 'list_dir', 'params': ['path?'], 'examples': [
        {'user': 'What files are in this project?', 'args': {}},
    ]},
    {'name': 'grep', 'params': ['pattern', 'path?', 'glob?'], 'examples': [
        {'user': 'Find all feature sets that use Emit', 'args': {'pattern': 'Emit', 'glob': '*.aro'}},
    ]},
    {'name': 'run_shell', 'params': ['command'], 'examples': [
        {'user': 'Show the git log', 'args': {'command': 'git log --oneline -5'}},
    ]},
    {'name': 'aro_check', 'params': ['path'], 'examples': [
        {'user': 'Check if my app has syntax errors', 'args': {'path': '.'}},
        {'user': 'Validate the code', 'args': {'path': '.'}},
    ]},
    {'name': 'aro_run', 'params': ['path', 'args?'], 'examples': [
        {'user': 'Run the app', 'args': {'path': '.'}},
    ]},
    {'name': 'aro_test', 'params': ['path'], 'examples': [
        {'user': 'Run the tests', 'args': {'path': '.'}},
    ]},
    {'name': 'aro_build', 'params': ['path'], 'examples': [
        {'user': 'Build a native binary', 'args': {'path': '.'}},
    ]},
    {'name': 'parse_aro', 'params': ['path'], 'examples': [
        {'user': 'Show me the AST', 'args': {'path': 'main.aro'}},
    ]},
    {'name': 'list_actions', 'params': [], 'examples': [
        {'user': 'What actions can I use?', 'args': {}},
    ]},
    {'name': 'create_plugin', 'params': ['name', 'language', 'handle?'], 'examples': [
        {'user': 'Create a Swift plugin for analytics', 'args': {'name': 'analytics', 'language': 'swift', 'handle': 'Analytics'}},
    ]},
    {'name': 'write_openapi', 'params': ['content'], 'examples': [
        {'user': 'Generate an OpenAPI contract for my user API', 'args': {'content': 'openapi: 3.0.3\ninfo:\n  title: User API\n  version: 1.0.0'}},
    ]},
    {'name': 'generate_docs', 'params': ['path'], 'examples': [
        {'user': 'Generate docs for my app', 'args': {'path': '.'}},
    ]},
    {'name': 'list_proposals', 'params': [], 'examples': [
        {'user': 'What language proposals exist?', 'args': {}},
    ]},
    {'name': 'read_proposal', 'params': ['number'], 'examples': [
        {'user': 'Show me the events proposal', 'args': {'number': '0007'}},
    ]},
    {'name': 'search_project', 'params': ['query', 'k?'], 'examples': [
        {'user': 'Find code related to authentication', 'args': {'query': 'user authentication login'}},
    ]},
]
print(f'{len(TOOLS)} tools, {sum(len(t["examples"]) for t in TOOLS)} examples')


## Generate training pairs

Three pair types:
1. **Direct call** — user asks → model calls the right tool
2. **Fix chains** — read → check → edit → verify (the `/fix` workflow)
3. **Tool selection** — given a task, which tool to use?

In [ ]:
import json as _json

def tool_call(name, args):
    return f'<tool_call>{_json.dumps({"name": name, "arguments": args})}</tool_call>'

pairs = []

# ---- Type 1: Direct tool calls ----
for tool in TOOLS:
    for ex in tool['examples']:
        pairs.append({
            'instruction': ex['user'],
            'output': tool_call(tool['name'], ex['args']),
            'source': f'fc_direct:{tool["name"]}',
            'task_type': 'function_calling',
            'score': 1.0,
        })
print(f'Direct calls: {len(pairs)}')

# ---- Type 2: Fix chains (aro ask /fix workflow) ----
FIX_SCENARIOS = [
    {'code': '(Application-Start: Demo) {\n    Compute the <msg> from "Hello" + <name>.\n    Log <msg> to the <console>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:45: error: Expected \'\'>\', but got +',
     'old': '"Hello" + <name>', 'new': '"Hello, " ++ <name>', 'desc': 'string concat uses ++'},
    {'code': '(Application-Start: Demo) {\n    Create the <count> with 0.\n    Compute the <count> from <count> + 1.\n    Return an <OK: status> for the <result>.\n}',
     'error': '3:18: error: Cannot rebind variable \'count\'',
     'old': 'Compute the <count>', 'new': 'Compute the <incremented>', 'desc': 'immutable variable'},
    {'code': '(Application-Start: Demo) {\n    Compute the <is-valid> from <age> > 18.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:22: error: Reserved prefix \'is-\'',
     'old': '<is-valid>', 'new': '<valid>', 'desc': 'reserved prefix'},
    {'code': '(Application-Start: Demo) {\n    Retrieve the <user> with the <user-repo>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:30: error: Expected preposition \'from\', but got \'with\'',
     'old': 'Retrieve the <user> with', 'new': 'Retrieve the <user> from', 'desc': 'wrong preposition'},
    {'code': '(Application-Start: Demo) {\n    Store the <user> from the <user-repo>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:24: error: Expected preposition \'into\', but got \'from\'',
     'old': 'Store the <user> from', 'new': 'Store the <user> into', 'desc': 'Store uses into'},
    {'code': '(Application-Start: Demo) {\n    Log <msg> for the <console>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:18: error: Expected preposition \'to\', but got \'for\'',
     'old': 'Log <msg> for the', 'new': 'Log <msg> to the', 'desc': 'Log uses to'},
    {'code': '(Application-Start: Demo) {\n    Emit the <user> to the <event>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:10: error: Expected event qualifier',
     'old': 'Emit the <user> to the <event>.', 'new': 'Emit a <UserCreated: event> with <user>.', 'desc': 'Emit syntax'},
    {'code': '(Application-Start: Demo) {\n    Extract the <id> from the <request: id>.\n    Return an <OK: status> for the <result>.\n}',
     'error': '2:40: error: Use pathParameters for URL params',
     'old': '<request: id>', 'new': '<pathParameters: id>', 'desc': 'path params'},
]

for s in FIX_SCENARIOS:
    # Full chain: user asks to fix -> read -> check -> edit -> verify
    # Step 1: user asks, model reads file
    pairs.append({'instruction': 'Fix the errors in main.aro',
                  'output': tool_call('read_file', {'path': 'main.aro'}),
                  'source': 'fc_fix:read', 'task_type': 'function_calling', 'score': 1.0})
    # Step 2: model sees code, runs check
    pairs.append({'instruction': f'File content:\n```aro\n{s["code"]}\n```\nCheck for errors.',
                  'output': tool_call('aro_check', {'path': '.'}),
                  'source': 'fc_fix:check', 'task_type': 'function_calling', 'score': 1.0})
    # Step 3: model sees error, fixes with edit
    pairs.append({'instruction': f'aro_check error:\n{s["error"]}\nFix it.',
                  'output': tool_call('edit_file', {'path': 'main.aro', 'old_string': s['old'], 'new_string': s['new']}),
                  'source': 'fc_fix:edit', 'task_type': 'function_calling', 'score': 1.0})
    # Step 4: verify
    pairs.append({'instruction': f'Fixed {s["desc"]}. Verify.',
                  'output': tool_call('aro_check', {'path': '.'}),
                  'source': 'fc_fix:verify', 'task_type': 'function_calling', 'score': 1.0})

print(f'Fix chains: {len(FIX_SCENARIOS) * 4}')

# ---- Type 3: Tool selection (which tool for this task?) ----
TOOL_SELECTION = [
    ('I want to see what files exist', 'list_dir'),
    ('Check my code for errors', 'aro_check'),
    ('Run my application', 'aro_run'),
    ('Find where the user model is defined', 'grep'),
    ('Create a new Rust plugin', 'create_plugin'),
    ('Generate API documentation', 'generate_docs'),
    ('Build a native binary of my app', 'aro_build'),
    ('What ARO actions are available?', 'list_actions'),
    ('Read the control flow proposal', 'read_proposal'),
    ('Search for authentication-related code', 'search_project'),
    ('Run the test suite', 'aro_test'),
    ('Show me the parse tree', 'parse_aro'),
    ('Write an openapi.yaml for my API', 'write_openapi'),
    ('Replace the old function name with the new one', 'edit_file'),
    ('Execute a git command', 'run_shell'),
]
for user_q, tool_name in TOOL_SELECTION:
    tool = next(t for t in TOOLS if t['name'] == tool_name)
    args = tool['examples'][0]['args'] if tool['examples'] else {}
    pairs.append({
        'instruction': user_q,
        'output': tool_call(tool_name, args),
        'source': f'fc_select:{tool_name}',
        'task_type': 'function_calling',
        'score': 1.0,
    })

print(f'Tool selection: {len(TOOL_SELECTION)}')
print(f'Total: {len(pairs)} function calling pairs')


## Prompt & output variations

Use the pre-warmed base model to generate 9 natural rephrasings of each
unique user-facing instruction. Each variation is paired with a contextual
output that wraps the `<tool_call>` in a natural assistant response, teaching
the model to embed tool calls in conversational flow.

In [ ]:
# ── Prompt & output variations ───────────────────────────────────────────────
# Use the pre-warmed model to generate 9 natural rephrasings of each unique
# user-facing instruction.  Pair each variation with a contextual output that
# embeds the tool call in a natural assistant response.
# ─────────────────────────────────────────────────────────────────────────────

import gc

# ── Load pre-warmed model ────────────────────────────────────────────────────
model, tokenizer, _load, generate, make_sampler = load_model()
sampler = make_sampler(temp=0.7, top_p=0.9)

# ── Output context templates ────────────────────────────────────────────────
# Each tool gets 9 natural lead-in phrases to wrap around the <tool_call>.
TOOL_CONTEXTS = {
    'read_file': [
        "Let me read that file.",
        "I'll pull up the file contents.",
        "Sure, let me take a look.",
        "Let me check what's in there.",
        "I'll open that up for you.",
        "Looking at the file now.",
        "Let me examine that.",
        "I'll read the file to see what we're working with.",
        "Sure, reading the file now.",
    ],
    'write_file': [
        "I'll create that file for you.",
        "Let me write that.",
        "Sure, here's a basic ARO application.",
        "I'll set that up now.",
        "Let me create the file.",
        "Writing the file now.",
        "I'll generate that for you.",
        "Let me put that together.",
        "Creating the file now.",
    ],
    'edit_file': [
        "I'll fix that for you.",
        "Let me apply the edit.",
        "Sure, I'll make that change.",
        "Applying the fix now.",
        "I see the issue — let me correct it.",
        "Let me update that.",
        "I'll patch that up.",
        "Making the edit now.",
        "Sure, here's the fix.",
    ],
    'list_dir': [
        "Let me list the project files.",
        "I'll show you what's in the directory.",
        "Sure, here's the file listing.",
        "Let me check the project structure.",
        "Looking at the directory contents.",
        "I'll see what files are here.",
        "Here's what's in the project.",
        "Let me scan the directory.",
        "Checking the file structure.",
    ],
    'grep': [
        "Let me search for that.",
        "I'll grep through the codebase.",
        "Searching the files now.",
        "Let me find that for you.",
        "I'll look through the source files.",
        "Searching for matches.",
        "Let me scan the code.",
        "I'll search the ARO files.",
        "Looking for that pattern now.",
    ],
    'run_shell': [
        "I'll run that command.",
        "Let me execute that.",
        "Sure, running the command now.",
        "I'll fire that off.",
        "Executing the command.",
        "Let me run that for you.",
        "Sure, here goes.",
        "Running the shell command.",
        "I'll execute that now.",
    ],
    'aro_check': [
        "Let me validate the code.",
        "I'll check for syntax errors.",
        "Running the syntax checker.",
        "Let me verify the code is correct.",
        "I'll run aro check on the project.",
        "Checking the code for errors.",
        "Let me lint the ARO code.",
        "Validating the syntax now.",
        "I'll scan for issues.",
    ],
    'aro_run': [
        "I'll run the application.",
        "Let me execute the app.",
        "Starting the application now.",
        "Running the ARO app.",
        "I'll fire up the application.",
        "Let me start it up.",
        "Executing the application.",
        "I'll launch the app.",
        "Running it now.",
    ],
    'aro_test': [
        "I'll run the tests.",
        "Let me execute the test suite.",
        "Running tests now.",
        "I'll check if the tests pass.",
        "Let me run the test suite.",
        "Executing the tests.",
        "I'll verify with the tests.",
        "Running the test suite.",
        "Let me see if the tests pass.",
    ],
    'aro_build': [
        "I'll build the native binary.",
        "Let me compile the application.",
        "Building the binary now.",
        "I'll compile it for you.",
        "Let me create the native build.",
        "Compiling the application.",
        "I'll generate the binary.",
        "Building the project.",
        "Let me build that.",
    ],
    'parse_aro': [
        "I'll parse the file and show the AST.",
        "Let me generate the parse tree.",
        "Parsing the file now.",
        "I'll show you the abstract syntax tree.",
        "Let me analyze the structure.",
        "Generating the AST.",
        "I'll parse that for you.",
        "Looking at the parse tree.",
        "Let me show the syntax tree.",
    ],
    'list_actions': [
        "I'll list all available actions.",
        "Let me show you what actions exist.",
        "Here are the available ARO actions.",
        "I'll pull up the action reference.",
        "Let me check the available actions.",
        "Listing all actions.",
        "I'll show the action catalog.",
        "Here's what you can use.",
        "Let me look up the actions.",
    ],
    'create_plugin': [
        "I'll scaffold the plugin for you.",
        "Let me create the plugin structure.",
        "Setting up the plugin now.",
        "I'll generate the plugin boilerplate.",
        "Creating the plugin.",
        "Let me set that up.",
        "I'll scaffold that plugin.",
        "Generating the plugin structure.",
        "Let me create the plugin project.",
    ],
    'write_openapi': [
        "I'll generate the OpenAPI contract.",
        "Let me create the API specification.",
        "Writing the openapi.yaml.",
        "I'll draft the API contract.",
        "Creating the OpenAPI spec.",
        "Let me write the contract.",
        "I'll generate the specification.",
        "Setting up the API contract.",
        "Let me create the openapi.yaml.",
    ],
    'generate_docs': [
        "I'll generate the documentation.",
        "Let me create the docs.",
        "Generating documentation now.",
        "I'll build the documentation.",
        "Creating the docs.",
        "Let me generate the docs.",
        "I'll produce the documentation.",
        "Generating the docs now.",
        "Let me document the project.",
    ],
    'list_proposals': [
        "I'll list the language proposals.",
        "Let me show you the proposals.",
        "Here are the ARO proposals.",
        "I'll pull up the proposal list.",
        "Listing the proposals.",
        "Let me check what proposals exist.",
        "I'll show the specifications.",
        "Here's the proposal catalog.",
        "Let me look up the proposals.",
    ],
    'read_proposal': [
        "I'll pull up that proposal.",
        "Let me read the proposal.",
        "Loading the proposal now.",
        "I'll show you that specification.",
        "Reading the proposal.",
        "Let me fetch that proposal.",
        "I'll display the proposal.",
        "Here's the proposal content.",
        "Let me look that up.",
    ],
    'search_project': [
        "I'll search the project for that.",
        "Let me look through the codebase.",
        "Searching the project now.",
        "I'll find relevant code.",
        "Let me search for related files.",
        "Searching for matches.",
        "I'll scan the project.",
        "Let me find that in the codebase.",
        "Looking through the project.",
    ],
}
GENERIC_CONTEXTS = [
    "Sure, let me handle that.",
    "I'll take care of that.",
    "On it.",
    "Let me do that for you.",
    "Sure, here you go.",
    "Working on that now.",
    "I'll handle that.",
    "Let me get that done.",
    "Sure, doing that now.",
]

# ── Filter: only variate user-facing instructions ────────────────────────────
def should_variate(instruction):
    """Skip model-internal prompts (contain code blocks or error traces)."""
    return ('```' not in instruction
            and not ('error:' in instruction.lower() and '\n' in instruction))

variatable_instrs = [p['instruction'] for p in pairs if should_variate(p['instruction'])]
unique_instructions = sorted(set(variatable_instrs))
skipped = len(set(p['instruction'] for p in pairs)) - len(unique_instructions)
print(f'Generating 9 variations for {len(unique_instructions)} unique instructions '
      f'(skipping {skipped} model-internal prompts)...\n')

variation_map = {}
for i, instr in enumerate(unique_instructions):
    prompt_text = (
        "/no_think\n"
        "Rephrase this user request in 9 different ways. Each should sound "
        "natural and express the same intent. Vary the style: questions, "
        "commands, casual, formal, short, and long forms. Preserve any "
        "specific filenames, paths, or technical terms.\n\n"
        f'Original: "{instr}"\n\n'
        "Return ONLY 9 variations, one per line, numbered 1-9."
    )
    messages = [{"role": "user", "content": prompt_text}]
    formatted = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    response = generate(model, tokenizer, prompt=formatted,
                        sampler=sampler, max_tokens=600)
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()

    variations = []
    for line in response.split('\n'):
        cleaned = re.sub(r'^\d+[\.\)\-]\s*', '', line.strip())
        cleaned = cleaned.strip('"').strip("'").strip('*').strip()
        if cleaned and cleaned != instr and 5 < len(cleaned) < 300:
            variations.append(cleaned)
    variation_map[instr] = variations[:9]

    if (i + 1) % 10 == 0 or i + 1 == len(unique_instructions):
        print(f'  {i+1}/{len(unique_instructions)}')

total_vars = sum(len(v) for v in variation_map.values())
print(f'\nGenerated {total_vars} variations '
      f'(avg {total_vars / max(len(variation_map), 1):.1f} per instruction)')

# ── Build varied pairs with contextual outputs ──────────────────────────────
seen = set((p['instruction'], p['output']) for p in pairs)
varied_pairs = []

for p in pairs:
    instr = p['instruction']
    if not should_variate(instr):
        continue
    original_output = p['output']
    source = p.get('source', '')

    # Resolve tool name for context lookup
    tool_name = source.split(':')[1] if ':' in source else ''
    if tool_name not in TOOL_CONTEXTS:
        m = re.search(r'"name":\s*"(\w+)"', original_output)
        if m:
            tool_name = m.group(1)

    contexts = TOOL_CONTEXTS.get(tool_name, GENERIC_CONTEXTS)
    variations = variation_map.get(instr, [])

    for j, var_instr in enumerate(variations):
        ctx = contexts[j % len(contexts)]
        varied_output = f"{ctx}\n\n{original_output}"
        key = (var_instr, varied_output)
        if key in seen:
            continue
        seen.add(key)
        varied_pairs.append({
            'instruction': var_instr,
            'output': varied_output,
            'source': f'{source}:var{j}',
            'task_type': 'function_calling',
            'score': 1.0,
        })

pairs.extend(varied_pairs)
print(f'Added {len(varied_pairs)} varied pairs → total: {len(pairs)}')

# ── Free model memory ────────────────────────────────────────────────────────
import mlx.core as mx
del model, tokenizer
gc.collect(); mx.metal.clear_cache()
print('Model unloaded.')

## Save and merge

In [ ]:
clean_notebook_pairs('NB11')
for p in pairs:
    p['notebook'] = 'NB11'
new_count = save_notebook_pairs('NB11', pairs)
print(f'Saved {new_count} function calling pairs to {PAIRS_FILE}')

own_file = DATA_OUT / 'function_calling_pairs.jsonl'
with open(own_file, 'w') as f:
    for p in pairs:
        f.write(json.dumps(p) + '\n')
print(f'Local copy: {own_file}')


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white', 'legend.framealpha': 1.0,
    'figure.facecolor': '#fafafa', 'axes.facecolor': '#f9f9f9',
    'savefig.facecolor': '#fafafa', 'savefig.bbox': 'tight', 'savefig.dpi': 150,
})
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '11_function_calling.png'

_tool_counts = Counter()
_type_counts = Counter()
for p in pairs:
    src = p.get('source', '')
    tool = src.split(':')[1] if ':' in src else 'other'
    _tool_counts[tool] += 1
    ptype = src.split(':')[0].replace('fc_', '')
    _type_counts[ptype] += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

_sorted = sorted(_tool_counts.items(), key=lambda x: -x[1])
_names = [t for t, _ in _sorted]
_vals = [c for _, c in _sorted]
x = np.arange(len(_names))
ax1.bar(x, _vals, color='#3498db', edgecolor='white', width=0.7)
for i, v in enumerate(_vals):
    ax1.text(i, v + 0.2, str(v), ha='center', fontsize=8, fontweight='bold')
ax1.set_xticks(list(x))
ax1.set_xticklabels(_names, rotation=45, ha='right', fontsize=7)
ax1.set_ylabel('Pairs')
ax1.set_title('Pairs per tool', fontsize=11, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

_ts = sorted(_type_counts.items(), key=lambda x: -x[1])
ax2.bar([t for t,_ in _ts], [c for _,c in _ts],
        color=['#2ecc71','#3498db','#e67e22','#9b59b6'][:len(_ts)], edgecolor='white')
for i, (t, c) in enumerate(_ts):
    ax2.text(i, c + 0.2, str(c), ha='center', fontsize=9, fontweight='bold')
ax2.set_ylabel('Pairs')
ax2.set_title('Pairs by type', fontsize=11, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

fig.suptitle(f'Function Calling Training \u2014 {len(pairs)} pairs, {len(TOOLS)} tools',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')


In [ ]:
import csv
from datetime import date as _date
_run_dir = Path('.') / 'run' / _date.today().isoformat()
csv_path = _run_dir / '11_function_calling.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['tool', 'pair_type', 'instruction_preview', 'output_preview'])
    for p in pairs:
        src = p.get('source', '')
        tool = src.split(':')[1] if ':' in src else 'other'
        ptype = src.split(':')[0].replace('fc_', '')
        w.writerow([tool, ptype, p['instruction'][:150], p['output'][:200]])
print(f'CSV: {csv_path} ({len(pairs)} rows)')
